In [45]:
import torch
import torch.nn as nn

In [58]:
EMBEDDING_DIN = 64
CONTEXT_LENGTH = 100
BATCH = 4
inputs = torch.rand(BATCH,CONTEXT_LENGTH,EMBEDDING_DIN)
inputs.shape,inputs[0].shape,inputs[0,0,:].shape

(torch.Size([4, 100, 64]), torch.Size([100, 64]), torch.Size([64]))

In [59]:
torch.transpose?

Docstring:
transpose(input, dim0, dim1) -> Tensor

Returns a tensor that is a transposed version of :attr:`input`.
The given dimensions :attr:`dim0` and :attr:`dim1` are swapped.

If :attr:`input` is a strided tensor then the resulting :attr:`out`
tensor shares its underlying storage with the :attr:`input` tensor, so
changing the content of one would change the content of the other.

If :attr:`input` is a :ref:`sparse tensor <sparse-docs>` then the
resulting :attr:`out` tensor *does not* share the underlying storage
with the :attr:`input` tensor.

If :attr:`input` is a :ref:`sparse tensor <sparse-docs>` with compressed
layout (SparseCSR, SparseBSR, SparseCSC or SparseBSC) the arguments
:attr:`dim0` and :attr:`dim1` must be both batch dimensions, or must
both be sparse dimensions. The batch dimensions of a sparse tensor are the
dimensions preceding the sparse dimensions.

.. note::
    Transpositions which interchange the sparse dimensions of a `SparseCSR`
    or `SparseCSC` layout tens

# 1 Simple Attention

In [60]:
class NaiveAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        values = (x@x.transpose(1,2))@x
        print(values.shape)

In [61]:
naive = NaiveAttention()
naive.eval()
with torch.no_grad():
    naive(inputs)

torch.Size([4, 100, 64])


# 2 Scaled attention with K,Q,V projections & single Head

In [87]:
class attentionKQV(torch.nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        # input shape -> [ B , T , D_in]
        self.d_out = d_out
        self.Q = nn.Parameter(torch.rand(d_in,d_out))
        self.K = nn.Parameter(torch.rand(d_in,d_out))
        self.V = nn.Parameter(torch.rand(d_in,d_out))
    
    def forward(self,x):
        q_proj = x@self.Q  # [ B , T , D_out]
        k_proj = x@self.K  # [ B , T , D_out]
        v_proj = x@self.V  # [ B , T , D_out]
        attention = torch.softmax(q_proj @ k_proj.transpose(1,2) / q_proj.shape[-1]**0.,dim=2)
        print(attention.shape,"attention shape")
        print(attention.sum())
        z_vector = attention@v_proj
        print(z_vector.shape , "context shape")
        return z_vector        

In [88]:
torch.manual_seed(123)
kqv = attentionKQV(64,16)
z = kqv(inputs)
print("\n")

torch.Size([4, 100, 100]) attention shape
tensor(400., grad_fn=<SumBackward0>)
torch.Size([4, 100, 16]) context shape




# 2.1 Scaled attention with K,Q,V projections & single Head , using Linear Layer

In [94]:
class attentionKQV_2(torch.nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        # input shape -> [ B , T , D_in]
        self.d_out = d_out
        self.Q = nn.Linear(d_in,d_out,bias=False)
        self.K = nn.Linear(d_in,d_out,bias=False)
        self.V = nn.Linear(d_in,d_out,bias=False)
        print(self.Q.weight.shape)
    
    def forward(self,x):
        q_proj = self.Q(x)  # [ B , T , D_out]  ( B , T , D_in)  -> [ B , T , D_out]
        k_proj = self.K(x)
        v_proj = self.V(x)
        print(q_proj.shape)
        attention = torch.softmax(q_proj @ k_proj.transpose(1,2) / q_proj.shape[-1]**0.,dim=2)
        print(attention.shape,"attention shape")
        print(attention.sum())
        z_vector = attention@v_proj
        print(z_vector.shape , "context shape")
        return z_vector        
         

In [95]:
torch.manual_seed(123)
kqv_2 = attentionKQV_2(64,16)
z = kqv_2(inputs)
print("\n")

torch.Size([16, 64])
torch.Size([4, 100, 16])
torch.Size([4, 100, 100]) attention shape
tensor(400., grad_fn=<SumBackward0>)
torch.Size([4, 100, 16]) context shape


